# Raw to Foundation layer load

Appends new raw rows to the foundation table, driven by `metadata_ict.etl_setup_config` (`target_layer = 'foundation'`). Watermark from `metadata_ict.etl_watermark` (`layer = 'foundation'`) on the raw `_ingest_ts`; each run writes a SUCCESS/FAILED row to `audit_ict.audit_task_run`.

In [0]:
%run ../Utilities/Audit_Utils

In [0]:
%python
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "ent_gf_dev")
dbutils.widgets.text("entity_id", "zipt_markup")

catalog = dbutils.widgets.get("catalog")
entity_id = dbutils.widgets.get("entity_id")
AUDIT_TASK_TABLE = f"{catalog}.audit_ict.audit_task_run"
WATERMARK_TABLE = f"{catalog}.metadata_ict.etl_watermark"

cfg = (
    spark.table(f"{catalog}.metadata_ict.etl_setup_config")
    .filter((F.col("entity_id") == entity_id) & (F.col("target_layer") == "foundation") & F.col("is_active"))
    .first()
)

source_table = f"{catalog}.{cfg['source_read_from']}"   # ent_gf_dev.r_ict_conf.zipt_markup
target_table = f"{catalog}.{cfg['target_table']}"       # ent_gf_dev.f_ict_conf.zipt_markup
partition_column = cfg["partition_column"]
print(f"{entity_id}: {source_table} -> {target_table} (append)")

In [0]:
%python
import uuid
from datetime import datetime, UTC
from delta.tables import DeltaTable

run_id = str(uuid.uuid4())
source_schema = source_table.rsplit(".", 1)[0]

# Watermark: empty last_load_type = initial load, otherwise incremental on raw _ingest_ts
wm = spark.table(WATERMARK_TABLE).filter((F.col("entity_id") == entity_id) & (F.col("layer") == "foundation")).first()
wm_col = wm["watermark_column"]   # _ingest_ts
is_initial = not wm["last_load_type"]
new_load_type = "initial" if is_initial else "incremental"

audit_args = dict(
    table_name=entity_id, run_id=run_id, layer=cfg["target_layer"], target_table=target_table,
    load_type="APPEND", source_schema=source_schema, start_ts=datetime.now(UTC),
    watermark_from=wm["last_watermark_value"],
)

try:
    raw_df = spark.table(source_table)
    max_wm = raw_df.agg(F.max(wm_col)).first()[0]   # highest raw _ingest_ts
    # Initial: all raw rows up to max(_ingest_ts); incremental: last_watermark_value < _ingest_ts <= max(_ingest_ts)
    window_df = raw_df.filter(F.col(wm_col) <= F.lit(max_wm))
    if not is_initial:
        window_df = window_df.filter(F.col(wm_col) > F.lit(wm["last_watermark_value"]))

    if not is_initial and window_df.isEmpty():
        # Nothing new since last watermark: skip append, watermark unchanged
        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), row_count=0,
            message=f"No new rows in {source_table} after {wm_col} {wm['last_watermark_value']}",
        ), AUDIT_TASK_TABLE)
        print(f"No new rows after {wm_col} {wm['last_watermark_value']}, load skipped")
    else:
        print(f"{new_load_type} load from {source_table}: {wm_col} > {wm['last_watermark_value']} and <= {max_wm}")

        # Foundation audit columns: one fixed _ingest_ts per run (used by trusted as its watermark)
        ingest_ts = datetime.now(UTC)
        df = window_df.withColumn("_ingest_ts", F.lit(ingest_ts)).withColumn("_ingest_date", F.to_date(F.col("_ingest_ts")))

        # Append only
        writer = df.write.format("delta").mode("append").option("mergeSchema", "true")
        if partition_column:
            writer = writer.partitionBy(partition_column)
        writer.saveAsTable(target_table)
        rows_written = DeltaTable.forName(spark, target_table).history(1).first()["operationMetrics"]["numOutputRows"]

        # Move watermark: previous = last, last = highest raw _ingest_ts loaded this run
        spark.sql(
            f"""UPDATE {WATERMARK_TABLE}
                SET previous_watermark_value = last_watermark_value,
                    last_watermark_value = CAST(:max_wm AS TIMESTAMP),
                    last_run_id = :last_run_id,
                    last_load_type = :last_load_type,
                    last_success_ts = current_timestamp(),
                    updated_by = current_user()
                WHERE entity_id = :entity_id AND layer = 'foundation'""",
            args={"max_wm": max_wm, "last_run_id": stable_long_id(run_id),
                  "last_load_type": new_load_type, "entity_id": entity_id},
        )

        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), row_count=rows_written, watermark_to=max_wm,
        ), AUDIT_TASK_TABLE)
        print(f"Appended {rows_written} rows {source_table} -> {target_table}, watermark = {max_wm}")

except Exception as e:
    write_task_audit(spark, build_task_audit_details(
        **audit_args, status="FAILED", end_ts=datetime.now(UTC),
        message=str(e)[:2000], rows_updated=None, rows_deleted=None,
    ), AUDIT_TASK_TABLE)
    raise